In [19]:
import requests
import json
import pandas as pd
import numpy as np
from pandas import json_normalize
import plotly.graph_objects as go
import plotly.express as px
import yfinance as yf

In [3]:
df = pd.read_csv("Stock_prices_from2012.06.csv", header=[0, 1], index_col=0, parse_dates=True)
df

Price            Close                                                  \
Ticker            AAPL        AMZN       GOOGL        META        MSFT   
Date                                                                     
2012-06-01   16.779005   10.411000   14.154103   27.478691   22.432699   
2012-06-04   16.877703   10.728500   14.342749   26.665831   22.511545   
2012-06-05   16.834030   10.660500   14.139973   25.644798   22.480003   
2012-06-06   17.092157   10.882000   14.391830   26.576614   23.142332   
2012-06-07   17.099934   10.940000   14.333823   26.080971   23.047722   
...                ...         ...         ...         ...         ...   
2026-09-10  326.570007  251.889999  332.600006  644.380005  492.440002   
2026-09-11  332.269989  256.779999  338.500000  648.030029  495.630005   
2026-09-14  333.079987  253.539993  349.390015  665.599976  505.410004   
2026-09-15  331.339996  248.419998  344.980011  670.239990  497.119995   
2026-09-16  332.410004  245.960007  342.869995  673.309998  490.299988   

Price                                              Volume              \
Ticker            NVDA         SPY        TSLA       AAPL        AMZN   
Date                                                                    
2012-06-01    0.273951  100.006142    1.876667  520987600  79030000.0   
2012-06-04    0.268234   99.959328    1.858667  556995600  85992000.0   
2012-06-05    0.276009  100.716232    1.860667  388214400  70878000.0   
2012-06-06    0.283326  102.979172    1.948000  401455600  54202000.0   
2012-06-07    0.271893  103.041573    1.928667  379766800  70078000.0   
...                ...         ...         ...        ...         ...   
2026-09-10  218.360001  757.830017  363.559998   70011900  25484800.0   
2026-09-11  218.289993  764.289978  365.440002   50716900  26724100.0   
2026-09-14  210.960007  760.880005  358.970001   39269100  34352800.0   
2026-09-15  212.169998  757.390015  356.579987   31748200  36275400.0   
2026-09-16  213.899994  754.049988  358.079987   35920400  33318500.0   

Price                                                                  \
Ticker            GOOGL        META      MSFT         NVDA        SPY   
Date                                                                    
2012-06-01  122193684.0  41855500.0  56634300  440984000.0  253240900   
2012-06-04   97210692.0  35230300.0  47926300  432856000.0  202545800   
2012-06-05   93502404.0  42473400.0  45715400  365224000.0  164149400   
2012-06-06   83748168.0  61489200.0  46860500  368968000.0  184202800   
2012-06-07   70269660.0  26159500.0  37792800  526780000.0  184772700   
...                 ...         ...       ...          ...        ...   
2026-09-10   23557600.0  21531900.0  16038800  105768000.0   42740400   
2026-09-11   24708300.0  16925000.0  14510500   89060100.0   45512700   
2026-09-14   35905400.0  19332300.0  23091900  132267200.0   43992400   
2026-09-15   21928600.0  19502800.0  17794600   88059700.0   46195000   
2026-09-16   18860000.0  17168100.0  16612700   96079300.0   58959700   

Price                   
Ticker            TSLA  
Date                    
2012-06-01  13287000.0  
2012-06-04  15463500.0  
2012-06-05   9463500.0  
2012-06-06  13648500.0  
2012-06-07   7381500.0  
...                ...  
2026-09-10  29667200.0  
2026-09-11  30153000.0  
2026-09-14  32477200.0  
2026-09-15  30317700.0  
2026-09-16  32031100.0  

[3593 rows x 16 columns]

In [11]:
df2 = df.drop(columns=["Volume"]).droplevel(0, axis=1)
df2


Ticker,AAPL,AMZN,GOOGL,META,MSFT,NVDA,SPY,TSLA
Date,,,,,,,,
2012-06-01,16.779005,10.411000,14.154103,27.478691,22.432699,0.273951,100.006142,1.876667
2012-06-04,16.877703,10.728500,14.342749,26.665831,22.511545,0.268234,99.959328,1.858667
2012-06-05,16.834030,10.660500,14.139973,25.644798,22.480003,0.276009,100.716232,1.860667
2012-06-06,17.092157,10.882000,14.391830,26.576614,23.142332,0.283326,102.979172,1.948000
2012-06-07,17.099934,10.940000,14.333823,26.080971,23.047722,0.271893,103.041573,1.928667
...,...,...,...,...,...,...,...,...
2026-09-10,326.570007,251.889999,332.600006,644.380005,492.440002,218.360001,757.830017,363.559998
2026-09-11,332.269989,256.779999,338.500000,648.030029,495.630005,218.289993,764.289978,365.440002
2026-09-14,333.079987,253.539993,349.390015,665.599976,505.410004,210.960007,760.880005,358.970001


In [13]:
normalized = ((df2 / df2.iloc[0])-1)*100
# Equal weight M7 average (ideal/theoretical distribution — 1/7 each)
normalized["M7"]= normalized[["GOOGL", "AMZN", "AAPL", "META", "MSFT","NVDA","TSLA"]].mean(axis=1)
# Simulating 1000 randomly weighted M7 portfolios and averaging to approximate realistic investor returns
mag7 = ["GOOGL", "AMZN", "AAPL", "META", "MSFT", "NVDA", "TSLA"]

n_simulations = 1
results = []

for _ in range(n_simulations):
    weights = np.random.dirichlet(np.ones(7))
    portfolio_return = (normalized[mag7] * weights).sum(axis=1)
    results.append(portfolio_return)

simulations = pd.DataFrame(results).T
simulations.index = normalized.index

# One average line
normalized["IRL-M7"] = simulations.mean(axis=1)
normalized.head()

Ticker,AAPL,AMZN,GOOGL,META,MSFT,NVDA,SPY,TSLA,M7,IRL-M7
Date,,,,,,,,,,
2012-06-01,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2012-06-04,0.588221,3.049660,1.332796,-2.958149,0.351478,-2.086798,-0.046811,-0.959148,-0.097420,0.231277
2012-06-05,0.327940,2.396497,-0.099834,-6.673873,0.210871,0.751251,0.710047,-0.852577,-0.562818,0.292166
2012-06-06,1.866334,4.524058,1.679564,-3.282823,3.163386,3.422334,2.972848,3.801044,2.167699,2.435613
2012-06-07,1.912679,5.081158,1.269737,-5.086561,2.741635,-0.751251,3.035244,2.770866,1.134038,1.503776


In [15]:
fig1 = px.line(normalized,x=normalized.index,y=["M7","SPY", "IRL-M7"])
fig1.update_layout(title=dict(text="M7 vs SPY",font=dict(size=24)),yaxis_title="Change(%)")
fig1.show()

In [16]:
fig4 = px.line(normalized,x=normalized.index,y=["GOOGL", "AMZN", "AAPL", "META", "MSFT","NVDA","TSLA"])
fig4.update_traces(opacity=0.7)
fig4.update_layout(title=dict(text="The magnificent seven's growth over-time",font=dict(size=24)))
fig4.show()

In [17]:
# Resample to monthly to make animation smoother and faster
normalized_yearly = normalized.resample("YE").last()

# Reshape to long format
normalized_long = normalized_yearly.reset_index().melt(
    id_vars="Date",
    var_name="Company",
    value_name="Return"
)

normalized_long["Date"] = normalized_long["Date"].dt.strftime("%Y-%m")

# Animate
fig5 = px.bar(normalized_long,
              x="Company",
              y="Return",
              animation_frame="Date",
              range_y=[0.01, normalized_long["Return"].max()],
              log_y=True,
              title="companies returns from 2012"
              )
fig5.show()

In [20]:
top7_2012 = ["AAPL", "XOM", "MSFT", "IBM", "GE", "CVX", "BRK-B"]
data_2012 = yf.download(top7_2012, start="2012-05-18", end="2026-09-16")["Close"]
print(data_2012.head())

[*********************100%***********************]  7 of 7 completed

Ticker           AAPL      BRK-B        CVX         GE         IBM       MSFT  \
Date                                                                            
2012-05-18  15.863461  78.910004  54.924278  70.383804  110.360786  23.079256   
2012-05-21  16.787672  79.800003  55.610432  71.015205  111.420036  23.457739   
2012-05-22  16.658772  79.650002  55.403996  71.238060  110.890442  23.465618   
2012-05-23  17.065243  79.750000  55.225521  71.238060  110.496056  22.953096   
2012-05-24  16.908514  79.800003  55.816826  71.498062  110.479103  22.921564   

Ticker            XOM  
Date                   
2012-05-18  46.572098  
2012-05-21  46.897926  
2012-05-22  46.846485  
2012-05-23  46.897926  
2012-05-24  47.223789  


In [21]:
normalized_1= ((data_2012/data_2012.iloc[0])-1)*100
normalized_1["M7"]= normalized_1[top7_2012].mean(axis= 1)
normalized_1.head()

Ticker,AAPL,BRK-B,CVX,GE,IBM,MSFT,XOM,M7
Date,,,,,,,,
2012-05-18,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2012-05-21,5.826040,1.127866,1.249272,0.897083,0.959806,1.639926,0.699622,1.771374
2012-05-22,5.013477,0.937774,0.873416,1.213711,0.479931,1.674066,0.589167,1.540220
2012-05-23,7.575789,1.064499,0.548469,1.213711,0.122570,-0.546637,0.699622,1.525432
2012-05-24,6.587803,1.127866,1.625051,1.583117,0.107209,-0.683263,1.399317,1.678157


In [22]:
fig6=go.Figure()
#first dataset(SPY)
fig6.add_scatter(x=normalized.index,y=normalized["SPY"],name="SPY",mode="lines")
#second dataset(past M7)
fig6.add_scatter(x=normalized_1.index,y=normalized_1["M7"],name="past M7",mode="lines")

fig6.show()


In [23]:
print("Current M7:", normalized["M7"].iloc[-1].round(2), "%")
print("2012 Top 7:", normalized_1["M7"].iloc[-1].round(2), "%")
print("SPY:", normalized["SPY"].iloc[-1].round(2), "%")

Current M7: 15408.9 %
2012 Top 7: 802.7 %
SPY: 654.0 %


## Part 3

In [24]:
individual_growth={}
for ticker in mag7:
    data=yf.download(ticker,period="max")["Close"].squeeze().dropna()
    normalized_ticker= ((data/data.iloc[0])-1 )*100
    individual_growth[ticker]= normalized_ticker
df_individual=pd.DataFrame(individual_growth)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [27]:
final_returns = df_individual.iloc[-1]

# Calculate years public
years = df_individual.apply(lambda x: x.dropna().shape[0] / 252).round(1)

fig7 = px.bar(x=final_returns.index, y=final_returns.values,
              title="Total Return Since IPO",
              labels={"x": "Company", "y": "% Return"},
              log_y=True,
              text=[f"{y} yrs" for y in years])

fig7.update_traces(textposition="outside")
fig7.show()

## Part 4

In [28]:
growth_velocity = ((1 + final_returns/100) ** (1/years) - 1) * 100
fig_velocity=px.bar(growth_velocity,x=growth_velocity.index,y=growth_velocity.values)
fig_velocity.show()

# Dan's work

In [4]:
change = df["Close"].pct_change()*100
change

Ticker,AAPL,AMZN,GOOGL,META,MSFT,NVDA,SPY,TSLA
Date,,,,,,,,
2012-06-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2012-06-04,0.588221,3.049660,1.332796,-2.958149,0.351478,-2.086798,-0.046811,-0.959148
2012-06-05,-0.258759,-0.633833,-1.413787,-3.828991,-0.140114,2.898535,0.757213,0.107603
2012-06-06,1.533366,2.077768,1.781176,3.633548,2.946302,2.651165,2.246847,4.693638
2012-06-07,0.045496,0.532987,-0.403057,-1.864961,-0.408819,-4.035477,0.060596,-0.992454
...,...,...,...,...,...,...,...,...
2026-09-10,3.561239,-0.202058,0.589751,-1.424222,0.160685,-2.264792,-0.599424,-1.155488
2026-09-11,1.745409,1.941323,1.773901,0.566440,0.647795,-0.032061,0.852429,0.517110
2026-09-14,0.243777,-1.261783,3.217139,2.711286,1.973246,-3.357912,-0.446162,-1.770469


In [5]:
change_sum = change.cumsum()
change_sum

Ticker,AAPL,AMZN,GOOGL,META,MSFT,NVDA,SPY,TSLA
Date,,,,,,,,
2012-06-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2012-06-04,0.588221,3.049660,1.332796,-2.958149,0.351478,-2.086798,-0.046811,-0.959148
2012-06-05,0.329462,2.415827,-0.080991,-6.787140,0.211364,0.811738,0.710401,-0.851545
2012-06-06,1.862828,4.493594,1.700185,-3.153592,3.157666,3.462903,2.957248,3.842093
2012-06-07,1.908324,5.026581,1.297128,-5.018553,2.748847,-0.572574,3.017844,2.849638
...,...,...,...,...,...,...,...,...
2026-09-10,353.845437,393.128695,370.857053,426.761069,359.428080,811.402765,222.334100,757.233502
2026-09-11,355.590846,395.070018,372.630954,427.327509,360.075875,811.370704,223.186529,757.750612
2026-09-14,355.834623,393.808235,375.848092,430.038795,362.049121,808.012792,222.740367,755.980143


### Instead of merging the two df (df3 and change) I am going to use them separately.


In [8]:
fig = px.line(change_sum, color='Ticker')

fig.update_layout(title={"text":"Percentage change of stocks over time", 'x':0.5, 'xanchor':'center'}, font=dict(size=16), 
        yaxis={"title": {"text": "Change %", "font": {"size": 14}}},
        xaxis={
            "title": {"text": "Date", "font": {"size": 14}}},
        height=600)
fig.show()

## This is a nice tool to filter from certain dates. 
if the date is not in the list, the next valid date is used.

In [208]:
def rebase(change, start, end):
    return change.loc[start:end].cumsum()

start_date = str(input())
end_date = str(input())
rebase(change, start_date, end_date)


Ticker,AAPL,AMZN,GOOGL,META,MSFT,NVDA,SPY,TSLA
Date,,,,,,,,
2015-01-02,-0.951255,-0.589659,-0.209176,0.551138,0.667387,0.399023,-0.053529,-1.393818
2015-01-05,-3.768403,-2.641388,-2.114567,-1.054961,-0.252202,-1.289995,-1.859483,-5.597920
2015-01-06,-3.759000,-4.924721,-4.582508,-2.402303,-1.719933,-4.321807,-2.801375,-5.031497
2015-01-07,-2.356796,-3.864747,-4.876593,-2.402303,-0.449422,-4.582396,-1.555297,-5.187690
2015-01-08,1.485455,-3.181145,-4.528185,0.263489,2.492400,-0.820614,0.219258,-5.344120
...,...,...,...,...,...,...,...,...
2025-12-24,287.180998,329.931237,293.486701,293.432989,290.874988,726.578750,157.186047,531.083509
2025-12-26,287.031275,329.991483,293.302046,292.794839,290.811473,727.596729,157.175906,528.980091
2025-12-29,287.162945,329.797953,293.317993,292.101329,290.686395,726.384320,156.819541,525.707718


## It can be immedieately called in for a plot. It sort of works as a time range checker.

In [213]:
rebased = rebase(change, start_date, end_date)
close = df3["Close"].loc[rebased.index]
daily  = change.loc[rebased.index]

fig4 = px.line(rebased, color='Ticker',
               labels={"value": "Change %", "index": "Date"})

for tr in fig4.data:
    tr.customdata = np.stack([
        daily[tr.name].to_numpy(),
        close[tr.name].to_numpy(),],
        axis = 1)
    tr.hovertemplate = (
        "<b>%{fullData.name}</b><br>"
        "Total Change: %{y:.2f}%<br>"
        "Daily Change: %{customdata[0]:.2f}%<br>" 
        "Close Price: $%{customdata[1]:.2f}<br>"
        "Date: %{x|%d %b %Y}<br>"
        "<extra></extra>"
    )

fig4.update_layout(
    title={"text": "Percentage change of stocks over time", 'x': 0.5, 'xanchor': 'center'},
    font=dict(size=16),
    yaxis={"title": {"text": "Change %", "font": {"size": 14}}},
    xaxis={"title": {"text": "Date", "font": {"size": 14}}, "hoverformat": "%d %b %Y"},
    height=600,
)
fig4.show()

## Lets check when did the market fell 10 or more %
as a follow up would be interesting to check the positive as well.

In [214]:
spy = df3["Close"]["SPY"]
dd = (spy / spy.cummax() - 1) * 100      # % below the highest point so far
dd

Date
2012-06-01    0.000000
2012-06-04   -0.046811
2012-06-05    0.000000
2012-06-06    0.000000
2012-06-07    0.000000
                ...   
2026-09-10   -2.577517
2026-09-11   -1.747060
2026-09-14   -2.185427
2026-09-15   -2.634081
2026-09-16   -3.063457
Name: SPY, Length: 3593, dtype: float64

In [215]:
fig5 = px.line(dd,)

fig5.update_traces(hovertemplate = (
        "<b>SnP 500</b><br>"
        "Change since last max = %{y:.2f}%<br>" 
        "Date: %{x|%d %b %Y}<br>"
        "<extra></extra>"))

fig5.update_layout(
    title={"text": "Percentage fall (drawdown) compared to previous values represented on the SnP 500", 'x': 0.5, 'xanchor': 'center'},
    font=dict(size=16),
    yaxis={"title": {"text": "Change %", "font": {"size": 16}}},
    xaxis={"title": {"text": "Date", "font": {"size": 16}}, "hoverformat": "%d %b %Y"},
    showlegend=False,
    height=600,
)


fig5.add_annotation(
    x="2015-08-25", y=-11.9,
    text="Panic over Chinese economy",
    showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1.5, arrowcolor="crimson",
    ax=-150, ay=-20,              # arrow tail offset in pixels
    bgcolor="white", bordercolor="black", borderwidth=1,
)

fig5.add_annotation(
    x="2016-02-11", y=-13.0,
    text="Fear of slowing economy",
    showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1.5, arrowcolor="crimson",
    ax=0, ay=50,              # arrow tail offset in pixels
    bgcolor="white", bordercolor="black", borderwidth=1,
)


fig5.add_annotation(
    x="2018-12-24", y=-19.4,
    text="US political tension",
    showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1.5, arrowcolor="crimson",
    ax=-100, ay=20,              # arrow tail offset in pixels
    bgcolor="white", bordercolor="black", borderwidth=1,
)


fig5.add_annotation(
    x="2020-03-23", y=-33.7,
    text="COVID crash",
    showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1.5,arrowcolor="crimson",
    ax=-100, ay=50,              # arrow tail offset in pixels
    bgcolor="white", bordercolor="black", borderwidth=1,
)

fig5.add_annotation(
    x="2025-4-08", y=-18.8,
    text="Chinese trade tension",
    showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1.5, arrowcolor="crimson",
    ax=50, ay=50,              # arrow tail offset in pixels
    bgcolor="white", bordercolor="black", borderwidth=1,
)


fig5.add_vrect(
    x0="2022-01-03", x1="2022-10-12",
    fillcolor="red", opacity=0.12, line_width=0,
    annotation_text="Fed hiking cycle", annotation_position="bottom",
)


fig5.show()

## Investment calculator - we shoudl combine it somehow with selecting specific stocks - or just calculating it for the best pick from the other dataset


In [216]:

#select a time and date between 2012-06 and 2026-09-15 for start and end
def yield_calc(change, start, end, invest):
    growth = (1 + change["SPY"].loc[start:end] / 100).prod()
    return invest * growth

y_start_date = (input())
y_end_date = (input())
amount = float(input())
yield_calc(change, y_start_date, y_end_date, amount)
print(f"By investing {amount:,.0f}€ from {y_start_date} until {y_end_date} would result you {yield_calc(change, y_start_date, y_end_date, amount):,.2f}€, which is {yield_calc(change, y_start_date, y_end_date, amount)-amount:,.2f}€ profit and it equals to {yield_calc(change, y_start_date, y_end_date, amount)/amount*100:,.2f}% yield")


By investing 10,000€ from 2015 until 2026 would result you 44,413.69€, which is 34,413.69€ profit and it equals to 444.14% yield


## Future projext - Would be interesting to calculate what happens if you try to time the market.


# Trading volumes vs price or price change


In [ ]:
change_vol = df3["Volume"].pct_change()*100
change_vol


Ticker,AAPL,AMZN,GOOGL,META,MSFT,NVDA,SPY,TSLA
Date,,,,,,,,
2012-06-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2012-06-04,6.911489,8.809313,-20.445404,-15.828744,-15.375841,-1.843151,-20.018528,16.380673
2012-06-05,-30.302071,-17.576054,-3.814691,20.559291,-4.613125,-15.624596,-18.956898,-38.801048
2012-06-06,3.410796,-23.527752,-10.432070,44.771080,2.504845,1.025124,12.216554,44.222539
2012-06-07,-5.402540,29.290432,-16.094093,-57.456757,-19.350412,42.771189,0.309387,-45.917134
...,...,...,...,...,...,...,...,...
2026-09-10,6.660420,-22.902996,-28.939780,-40.023732,24.432100,27.499684,30.256854,-8.896552
2026-09-11,-27.559601,4.862899,4.884623,-21.395697,-9.528768,-15.796744,6.486369,1.637499
2026-09-14,-22.571963,28.546144,45.317161,14.223338,59.139244,48.514542,-3.340386,7.708022


# This is probably a not great graph

In [235]:
vol = df3["Volume"]
roll_vol_med = vol.rolling(63).median()
test = vol-roll_vol_med
test

Ticker,AAPL,AMZN,GOOGL,META,MSFT,NVDA,SPY,TSLA
Date,,,,,,,,
2012-06-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2012-06-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2012-06-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2012-06-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2012-06-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
2026-09-10,24279300.0,-10443000.0,-1789600.0,5345000.0,-13258700.0,-22595500.0,-1180500.0,-7414200.0
2026-09-11,4577000.0,-9038300.0,-527600.0,738100.0,-14696800.0,-36634000.0,1591800.0,-6582400.0
2026-09-14,-6870800.0,-1379100.0,10558200.0,2779600.0,-5909600.0,3903700.0,71500.0,-4153400.0


In [236]:


fig7 = px.line(test, color='Ticker')

fig7.update_layout(title={"text":"Percentage change of stock volumes traded over time", 'x':0.5, 'xanchor':'center'}, font=dict(size=16), 
        yaxis={"title": {"text": "Volume Change %", "font": {"size": 14}}},
        xaxis={
            "title": {"text": "Date", "font": {"size": 14}}},
        height=600)
fig7.show()

In [220]:
fig8 = px.line(change, color='Ticker')
fig8.update_layout(title={"text":"Price percentage change over time", 'x':0.5, 'xanchor':'center'}, font=dict(size=16), 
        yaxis={"title": {"text": "Change %", "font": {"size": 14}}},
        xaxis={
            "title": {"text": "Date", "font": {"size": 14}}},
        height=600)
fig8.show()

In [ ]:
from plotly.subplots import make_subplots
price = df3["Close"]
vol = df3["Volume"]

tickers=list(price.columns)

fig10 = make_subplots(rows=3, cols=3, subplot_titles = tickers, horizontal_spacing=0.07, vertical_spacing=0.09)

for i, t in enumerate(tickers):
    r, c = divmod(i, 3)
    x = price[t]
    y = vol[t]

    ok = x.notna() & y.notna() & (y > 0)
    logy = np.log10(y[ok])
    slope, intercept = np.polyfit(x[ok], logy, 1)
    corr = np.corrcoef(x[ok], logy)[0, 1]

    xs = np.linspace(x[ok].min(), x[ok].max(), 100)
    ys = 10 ** (slope * xs + intercept)

    fig10.add_trace(
        go.Scatter(
            x=x, y=y, mode="markers",
            marker=dict(size=3, opacity=0.3),
            showlegend=False,
        ),
        row=r + 1, col=c + 1,
    )

    fig10.add_trace(
        go.Scatter(x=xs, y=ys, mode="lines",
                   line=dict(color="red", width=2),
                   showlegend=False, hoverinfo="skip"),
        row=r + 1, col=c + 1,
    )

    fig10.add_annotation(
        text=f"R²={corr**2:.2f}", x=0.05, y=0.95,
        xref="x domain", yref="y domain", showarrow=False,
        font=dict(size=12, color="red"),
        row=r + 1, col=c + 1,
    )

fig10.update_layout(height=900, title={"text":"Traded volume vs price", 'x':0.5, 'xanchor':'center'}, font=dict(size=16),
    hovermode=False)
fig10.update_yaxes(type="log")
fig10.show()

# Checking the same sort of trend but with the percentage change compared to volume change


In [238]:
rel_vol = vol / vol.rolling(63).median()

long = (change.stack().rename("change").to_frame()
        .join(rel_vol.stack().rename("rel_vol"))
        .join(vol.stack().rename("volume"))
        .dropna().reset_index())

surge = long[long["rel_vol"] >= 2.5].copy()     # days with 2x+ normal volume

fig12 = px.scatter(
    surge, x="Date", y="change",
    size="rel_vol", color="rel_vol",
    color_continuous_scale="Viridis",
    facet_col="Ticker", facet_col_wrap=2,
    size_max=18, opacity=0.7, height=1000,
    labels={"change": "Daily change (%)", "rel_vol": "Volume ÷ 63d median"},
    hover_data={"Date": "|%d %b %Y", "change": ":.2f", "rel_vol": ":.1f", "volume": ":,.0f"},
    title="High-volume days: when they happened, which way price moved, how big the surge",
)
fig12.add_hline(y=0, line_dash="dash", line_color="grey")
fig12.show()